# Sidequest: Equilibrium vs Disequilibrium Chemistry Survey

This notebook follows the same mechanics as Emilie's FRECKLL tutorial notebook, adapted to a
10-planet survey with sampled metallicity and C/O ratio, and three Kzz cases (low/medium/high)
instead of her two. Isothermal profiles only for now — non-isothermal comes later.

**Goal:** for each of 10 planets, on a shared pressure grid, compute:
- the equilibrium VMR (via ACE)
- the disequilibrium VMR at low, medium, and high Kzz (via FRECKLL's Rosenbrock solver)

Every design choice that is *not* directly taken from Emilie's notebook is flagged explicitly.

## Cell 1 — Imports and warning filters

This matches Emilie's notebook exactly. Solving thousands of ODE steps triggers three warnings
on every call, all of which are expected noise rather than errors:

- `MatrixRankWarning` — the Rosenbrock step's linear solve occasionally hits a singular Jacobian
  on a step that gets rejected and retried with a smaller step size; not fatal on its own.
- `UserWarning: 'where' used without 'out'` — an internal `np.divide(..., where=...)` call in
  FRECKLL doesn't pass `out=`, so numpy warns about uninitialized memory in masked-out entries.
  Harmless since FRECKLL never reads those entries.
- `RuntimeWarning: divide by zero` — FRECKLL's convergence test divides by the time difference
  between two saved steps; if a step got rejected and retried, that difference can be zero. The
  resulting `inf`/`nan` correctly *fails* the convergence check rather than lying about it.

**Caveat carried over from Emilie's notebook:** if a solve ever comes back with
`result["success"] == False` alongside a flood of these warnings, that's the one case where it
isn't just noise — the solver genuinely struggled and it's worth investigating.

In [1]:
import warnings
import numpy as np
from astropy import units as u

from scipy.sparse.linalg import MatrixRankWarning

warnings.filterwarnings("ignore", category=MatrixRankWarning)
warnings.filterwarnings("ignore", message=r"'where' used without 'out'", category=UserWarning)
warnings.filterwarnings("ignore", message=r"divide by zero encountered in scalar divide", category=RuntimeWarning)

print("Imports OK")

Imports OK


## Cell 2 — Shared pressure grid

This is the grid **every planet, every Kzz case, both equilibrium and disequilibrium** will
share, per your requirement that it matches your existing equilibrium dataset.

- `np.logspace(2, -8, 100)` creates 100 values evenly spaced *in log-space* from 10² down to
  10⁻⁸. Atmospheric pressure spans many orders of magnitude from the deep atmosphere to the top,
  so log-spacing gives even resolution across that whole range rather than clustering points at
  one end.
- `<< u.bar` is astropy's "attach a unit" operator (used throughout Emilie's notebook). It labels
  the array as being in bar without converting any values — a pure annotation that lets FRECKLL
  and astropy catch unit-mismatch bugs automatically.

In [2]:
# Pressure grid: log_P from -8 to 2 (bar), 100 layers
# High pressure (10^2 bar) = deep atmosphere, low pressure (10^-8 bar) = top of atmosphere
pressure = np.logspace(2, -8, 100) << u.bar

print("Pressure grid shape:", pressure.shape)
print("Pressure range:", pressure.min(), "to", pressure.max())

Pressure grid shape: (100,)
Pressure range: 1e-08 bar to 100.0 bar


## Cell 3 — Elemental abundance helper + planet sampling

**Flag:** the abundance-conversion function below is a standard literature convention that I
constructed for this notebook — it is *not* from Emilie's notebook or the FRECKLL docs, since
her notebook only ever uses one fixed composition. Everything else (the ACE call itself,
Rosenbrock, `solve()` arguments) will stay exactly matched to her notebook in later cells.

**How the conversion works:**
- `SOLAR_LOG_ABUND` holds the same solar reference log-abundances Emilie's notebook uses for
  H, He, C, N, O (dex values, with H fixed as the reference point at 12).
- Metallicity scales the "metal" elements (C, N, O) by a linear multiplier — e.g. 10x
  metallicity means 10x as many C, N, and O atoms relative to H.
- C/O ratio is imposed by first computing the *combined* C+O budget (after metallicity
  scaling), then splitting that budget between C and O so their ratio matches the target,
  while keeping the total C+O amount fixed.
- Nitrogen is scaled by metallicity alone (no ratio constraint requested for it).

**Planet sampling:**
- `rng = np.random.default_rng(42)` uses the same fixed seed as Emilie's notebook, so the
  "random" planets are reproducible on every re-run.
- Teq, Rp, Mp, Rstar, and C/O are drawn uniformly across your specified ranges.
- Metallicity is drawn *log-uniformly* instead of uniformly, since your range (0.1 to 10,000x)
  spans 5 orders of magnitude — plain uniform sampling would put almost all planets above
  ~500x solar. This was flagged to you as a judgment call, not a given.

In [3]:
# --- Elemental abundance helper (my construction, not from Emilie's notebook) ---
# Solar reference log-abundances, same values used in Emilie's notebook
SOLAR_LOG_ABUND = {"H": 12.0, "He": 10.93, "C": 8.39, "N": 7.86, "O": 8.73}

def elemental_abundances(metallicity, co_ratio):
    """Return elements list + log-abundances list for ACE, given metallicity (xSolar) and C/O ratio."""
    n_H = 10 ** SOLAR_LOG_ABUND["H"]
    n_He = 10 ** SOLAR_LOG_ABUND["He"]
    n_C_solar = 10 ** SOLAR_LOG_ABUND["C"]
    n_N_solar = 10 ** SOLAR_LOG_ABUND["N"]
    n_O_solar = 10 ** SOLAR_LOG_ABUND["O"]

    # scale metals by metallicity
    n_N = n_N_solar * metallicity
    metal_CO_budget = (n_C_solar + n_O_solar) * metallicity

    # redistribute C+O budget to match target C/O ratio
    n_C = metal_CO_budget * (co_ratio / (1 + co_ratio))
    n_O = metal_CO_budget * (1 / (1 + co_ratio))

    elements = ["H", "He", "C", "N", "O"]
    abundances = [
        np.log10(n_H),
        np.log10(n_He),
        np.log10(n_C),
        np.log10(n_N),
        np.log10(n_O),
    ]
    return elements, abundances


# --- Sample 10 planets ---
rng = np.random.default_rng(42)  # matches Emilie's fixed seed, for reproducibility
n_planets = 10

param_ranges = {
    "Teq_K": (300.006, 2999.985),
    "Rp_Rj": (0.100, 2.000),
    "Mp_Mj": (0.011, 5.000),
    "Rstar_Rs": (0.500, 2.000),
    "CO_ratio": (0.20, 2.00),
    "Metallicity": (0.1, 10000.0),
}

planets = {}
planets["Teq_K"] = rng.uniform(*param_ranges["Teq_K"], size=n_planets)
planets["Rp_Rj"] = rng.uniform(*param_ranges["Rp_Rj"], size=n_planets)
planets["Mp_Mj"] = rng.uniform(*param_ranges["Mp_Mj"], size=n_planets)
planets["Rstar_Rs"] = rng.uniform(*param_ranges["Rstar_Rs"], size=n_planets)
planets["CO_ratio"] = rng.uniform(*param_ranges["CO_ratio"], size=n_planets)
# metallicity sampled log-uniformly since range spans 5 orders of magnitude
planets["Metallicity"] = 10 ** rng.uniform(
    np.log10(param_ranges["Metallicity"][0]),
    np.log10(param_ranges["Metallicity"][1]),
    size=n_planets,
)

for i in range(n_planets):
    print(
        f"Planet {i:2d}: Teq={planets['Teq_K'][i]:7.1f} K  "
        f"Rp={planets['Rp_Rj'][i]:.2f} Rjup  Mp={planets['Mp_Mj'][i]:.3f} Mjup  "
        f"Rstar={planets['Rstar_Rs'][i]:.2f} Rsun  "
        f"C/O={planets['CO_ratio'][i]:.2f}  Met={planets['Metallicity'][i]:8.2f}xSolar"
    )

Planet  0: Teq= 2389.7 K  Rp=0.80 Rjup  Mp=3.793 Mjup  Rstar=1.62 Rsun  C/O=0.99  Met=    1.00xSolar
Planet  1: Teq= 1485.0 K  Rp=1.86 Rjup  Mp=1.780 Mjup  Rstar=1.95 Rsun  C/O=1.70  Met=    0.11xSolar
Planet  2: Teq= 2618.2 K  Rp=1.32 Rjup  Mp=4.854 Mjup  Rstar=0.99 Rsun  C/O=1.46  Met=  860.24xSolar
Planet  3: Teq= 2182.9 K  Rp=1.66 Rjup  Mp=4.467 Mjup  Rstar=1.06 Rsun  C/O=0.76  Met=  210.99xSolar
Planet  4: Teq=  554.3 K  Rp=0.94 Rjup  Mp=3.894 Mjup  Rstar=1.20 Rsun  C/O=1.70  Met=  335.60xSolar
Planet  5: Teq= 2934.2 K  Rp=0.53 Rjup  Mp=0.982 Mjup  Rstar=0.78 Rsun  C/O=1.65  Met=  801.02xSolar
Planet  6: Teq= 2355.1 K  Rp=1.15 Rjup  Mp=2.339 Mjup  Rstar=0.69 Rsun  C/O=0.90  Met=   19.71xSolar
Planet  7: Teq= 2422.4 K  Rp=0.22 Rjup  Mp=0.230 Mjup  Rstar=1.21 Rsun  C/O=0.72  Met=   69.78xSolar
Planet  8: Teq=  645.9 K  Rp=1.67 Rjup  Mp=0.781 Mjup  Rstar=0.84 Rsun  C/O=1.43  Met=    0.50xSolar
Planet  9: Teq= 1516.0 K  Rp=1.30 Rjup  Mp=3.419 Mjup  Rstar=1.50 Rsun  C/O=0.45  Met=    0